<a href="https://colab.research.google.com/github/Evgmakoec/gold---trading---bot/blob/main/Crypto_Forex_CFD_Signal_Bot_.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:


import requests
import getpass
import time

print("Crypto Signal Bot — Channel ID Finder")
TOKEN = getpass.getpass("Paste your Telegram bot token here (private): ")

print("\nNow send a NEW message in your Crypto Forex CFD Signals channel.")
print("Waiting up to 90 seconds...\n")

url = f"https://api.telegram.org/bot{TOKEN}/getUpdates"
deadline = time.time() + 90
offset = None
found = False

while time.time() < deadline:
    params = {"timeout": 10, "allowed_updates": ["channel_post"]}
    if offset is not None:
        params["offset"] = offset

    try:
        response = requests.get(url, params=params, timeout=15)
        data = response.json()

        if not data.get("ok"):
            print("Telegram error:", data.get("description"))
            break

        for update in data.get("result", []):
            offset = update["update_id"] + 1
            post = update.get("channel_post")

            if post:
                chat = post["chat"]
                if chat.get("type") == "channel":
                    print("CHANNEL FOUND!")
                    print("Channel name:", chat.get("title"))
                    print("Channel ID:", chat["id"])
                    found = True
                    break

        if found:
            break

    except Exception as e:
        print("Connection error:", e)
        break

if not found:
    print("Channel not found yet.")
    print("Check that your bot is an administrator, then try again.")

Crypto Signal Bot — Channel ID Finder
Paste your Telegram bot token here (private): ··········

Now send a NEW message in your Crypto Forex CFD Signals channel.
Waiting up to 90 seconds...

CHANNEL FOUND!
Channel name: Crypto forex cfd signals
Channel ID: -1004374777759


In [ ]:



import requests
import getpass

TOKEN = getpass.getpass("Enter your Telegram bot token: ")
CHANNEL_ID = input("Enter your channel ID: ").strip()

url = f"https://api.telegram.org/bot{TOKEN}/sendMessage"
response = requests.post(
    url,
    json={
        "chat_id": CHANNEL_ID,
        "text": "✅ Crypto Signal Bot connected!\n\nTelegram alerts are working. Automatic trading is OFF."
    },
    timeout=20
)

print(response.json())

Enter your Telegram bot token: ··········
Enter your channel ID: -1004374777759
{'ok': True, 'result': {'message_id': 9, 'sender_chat': {'id': -1004374777759, 'title': 'Crypto forex cfd signals', 'type': 'channel'}, 'chat': {'id': -1004374777759, 'title': 'Crypto forex cfd signals', 'type': 'channel'}, 'date': 1791617014, 'text': '✅ Crypto Signal Bot connected!\n\nTelegram alerts are working. Automatic trading is OFF.'}}


In [ ]:


import requests
import pandas as pd

BASE_URLS = [
    "https://data-api.binance.vision",
    "https://api.binance.com"
]

COINS = ["BTCUSDT", "ETHUSDT", "SOLUSDT"]
TIMEFRAMES = ["4h", "1h", "5m"]

def get_candles(symbol, timeframe):
    for base_url in BASE_URLS:
        try:
            url = f"{base_url}/api/v3/klines"
            response = requests.get(
                url,
                params={
                    "symbol": symbol,
                    "interval": timeframe,
                    "limit": 100
                },
                timeout=15
            )
            response.raise_for_status()
            data = response.json()

            if not isinstance(data, list) or len(data) < 3:
                continue

            df = pd.DataFrame(data, columns=[
                "time", "open", "high", "low", "close",
                "volume", "close_time", "quote_volume",
                "trades", "taker_buy_volume",
                "taker_buy_quote_volume", "ignore"
            ])

            # Exclude the newest candle because it may still be forming.
            df = df.iloc[:-1].copy()
            df["time"] = pd.to_datetime(
                df["time"], unit="ms", utc=True
            )
            for column in ["open", "high", "low", "close", "volume"]:
                df[column] = df[column].astype(float)

            return df

        except Exception:
            continue

    raise ConnectionError(
        f"Could not retrieve {symbol} {timeframe} data."
    )

print("CRYPTO MARKET DATA CONNECTION TEST")
print("-" * 35)

for coin in COINS:
    for timeframe in TIMEFRAMES:
        try:
            candles = get_candles(coin, timeframe)
            latest = candles.iloc[-1]

            print(
                f"{coin} | {timeframe} | "
                f"Last closed candle: {latest['time']} | "
                f"Close: {latest['close']}"
            )
        except Exception as error:
            print(f"{coin} | {timeframe} | ERROR: {error}")

print("\nData test complete. No trades were placed.")

CRYPTO MARKET DATA CONNECTION TEST
-----------------------------------
BTCUSDT | 4h | Last closed candle: 2026-10-10 00:00:00+00:00 | Close: 82596.83
BTCUSDT | 1h | Last closed candle: 2026-10-10 06:00:00+00:00 | Close: 82738.01
BTCUSDT | 5m | Last closed candle: 2026-10-10 07:35:00+00:00 | Close: 82745.0
ETHUSDT | 4h | Last closed candle: 2026-10-10 00:00:00+00:00 | Close: 2491.38
ETHUSDT | 1h | Last closed candle: 2026-10-10 06:00:00+00:00 | Close: 2493.33
ETHUSDT | 5m | Last closed candle: 2026-10-10 07:35:00+00:00 | Close: 2492.59
SOLUSDT | 4h | Last closed candle: 2026-10-10 00:00:00+00:00 | Close: 109.69
SOLUSDT | 1h | Last closed candle: 2026-10-10 06:00:00+00:00 | Close: 109.85
SOLUSDT | 5m | Last closed candle: 2026-10-10 07:35:00+00:00 | Close: 109.83

Data test complete. No trades were placed.


In [ ]:


print("TIMEFRAME TREND DIAGNOSTIC")
print("=" * 35)

for coin in COINS:
    print(f"\n{coin}")

    for timeframe in ["4h", "1h", "5m"]:
        try:
            df = get_candles(coin, timeframe)
            df = add_indicators(df)
            last = df.iloc[-1]

            if last["close"] > last["ema20"] > last["ema50"]:
                direction = "BULLISH"
            elif last["close"] < last["ema20"] < last["ema50"]:
                direction = "BEARISH"
            else:
                direction = "MIXED"

            print(
                f"  {timeframe}: {direction} | "
                f"Close={last['close']:.4f} | "
                f"EMA20={last['ema20']:.4f} | "
                f"EMA50={last['ema50']:.4f}"
            )

        except Exception as error:
            print(f"  {timeframe}: ERROR — {error}")

TIMEFRAME TREND DIAGNOSTIC

BTCUSDT
  4h: BEARISH | Close=82596.8300 | EMA20=83083.7679 | EMA50=83825.0825
  1h: MIXED | Close=82738.0100 | EMA20=82632.7775 | EMA50=82739.4026
  5m: BULLISH | Close=82745.0000 | EMA20=82718.1673 | EMA50=82708.4080

ETHUSDT
  4h: BEARISH | Close=2491.3800 | EMA20=2536.7340 | EMA50=2602.3270
  1h: MIXED | Close=2493.3300 | EMA20=2492.0735 | EMA50=2509.6115
  5m: BEARISH | Close=2492.5900 | EMA20=2494.3543 | EMA50=2494.3612

SOLUSDT
  4h: BEARISH | Close=109.6900 | EMA20=112.6471 | EMA50=115.7455
  1h: MIXED | Close=109.8500 | EMA20=109.8146 | EMA50=111.0979
  5m: MIXED | Close=109.8300 | EMA20=109.8973 | EMA50=109.8701


In [ ]:


import requests
import getpass
from datetime import datetime, timezone

# Reuse credentials already entered, if available.
if "TOKEN" not in globals() or not TOKEN:
    TOKEN = getpass.getpass("Enter your Telegram bot token: ")

if "CHANNEL_ID" not in globals() or not CHANNEL_ID:
    CHANNEL_ID = input("Enter your channel ID (include -100): ").strip()

if "sent_alerts" not in globals():
    sent_alerts = set()


def send_signal_to_telegram(result):
    if result.get("signal") not in ("BUY", "SELL"):
        return False

    coin = result["coin"]
    signal = result["signal"]
    candle_time = result.get("candle_time_utc", "")
    signal_key = (coin, signal, candle_time)

    if signal_key in sent_alerts:
        print(f"{coin}: duplicate alert skipped")
        return False

    live_price = result.get("live_price", "Unavailable")
    entry_status = result.get(
        "entry_status",
        "NOT CHECKED — verify price manually"
    )

    message = (
        f"🚨 CRYPTO DEMO SIGNAL — {signal}\n\n"
        f"Coin: {coin}\n"
        f"Setup: {result.get('setup_type', result.get('setup', 'N/A'))}\n"
        f"Entry reference: {result.get('entry_reference', 'N/A')}\n"
        f"Live price at analysis: {live_price}\n"
        f"Entry status: {entry_status}\n\n"
        f"Stop-loss reference: {result.get('stop_loss_reference', 'N/A')}\n"
        f"Take-profit reference: {result.get('take_profit_reference', 'N/A')}\n"
        f"Risk/reward: {result.get('risk_reward', 'N/A')}\n\n"
        f"4H trend: {result.get('trend_4h', 'N/A')}\n"
        f"1H trend: {result.get('trend_1h', 'N/A')}\n"
        f"Signal candle (UTC): {candle_time}\n\n"
        "⚠️ Demo alert only. Live price can change before you act. "
        "Verify price and risk. Automatic trading is OFF."
    )

    response = requests.post(
        f"https://api.telegram.org/bot{TOKEN}/sendMessage",
        json={"chat_id": CHANNEL_ID, "text": message},
        timeout=20
    )
    response.raise_for_status()
    data = response.json()

    if not data.get("ok"):
        raise RuntimeError(f"Telegram rejected the message: {data}")

    sent_alerts.add(signal_key)
    print(f"{coin}: Telegram alert sent")
    return True


print("Scanning crypto markets...")

for coin in COINS:
    result = analyze_coin(coin)
    print(
        f"{coin}: {result.get('signal')} — "
        f"{result.get('reason', result.get('setup', ''))}"
    )

    if result.get("signal") in ("BUY", "SELL"):
        try:
            send_signal_to_telegram(result)
        except Exception as error:
            print(f"{coin}: Telegram send failed — {error}")

print("Scan complete. No trades were placed.")

Scanning crypto markets...
BTCUSDT: WAIT — No valid 5M entry; 4H=SELL, 1H=MIXED
ETHUSDT: WAIT — No valid 5M entry; 4H=SELL, 1H=MIXED
SOLUSDT: WAIT — No valid 5M entry; 4H=SELL, 1H=MIXED
Scan complete. No trades were placed.


In [ ]:


test_result = {
    "coin": "BTCUSDT",
    "signal": "BUY",
    "setup": "Telegram integration test (NOT a trade)",
    "entry_reference": "TEST ONLY",
    "stop_loss_reference": "TEST ONLY",
    "take_profit_reference": "TEST ONLY",
    "risk_reward": "Not applicable",
    "trend_4h": "TEST",
    "trend_1h": "TEST",
    "candle_time_utc": datetime.now(timezone.utc).isoformat()
}

# Send a clearly labelled integration test.
send_signal_to_telegram(test_result)

BTCUSDT: Telegram alert sent


True

In [ ]:


import pandas as pd
import numpy as np
import requests

RISK_REWARD = 1.5
BREAKOUT_LOOKBACK = 20
MIN_BODY_ATR = 0.5

# Entry timing thresholds, measured using 5M ATR
ENTRY_NEAR_ATR = 0.25
ENTRY_INVALID_ATR = 0.5


def add_indicators(df):
    df = df.copy()
    df["ema20"] = df["close"].ewm(span=20, adjust=False).mean()
    df["ema50"] = df["close"].ewm(span=50, adjust=False).mean()

    prev_close = df["close"].shift(1)
    tr = pd.concat([
        df["high"] - df["low"],
        (df["high"] - prev_close).abs(),
        (df["low"] - prev_close).abs()
    ], axis=1).max(axis=1)

    df["atr"] = tr.rolling(14).mean()
    return df


def trend_direction(df):
    last = df.iloc[-1]

    if pd.isna(last["atr"]):
        return "WAIT"

    if last["close"] > last["ema20"] > last["ema50"]:
        return "BUY"

    if last["close"] < last["ema20"] < last["ema50"]:
        return "SELL"

    return "MIXED"


def get_live_price(symbol):
    """Get a current public market quote from Binance."""
    last_error = None

    for base_url in BASE_URLS:
        try:
            response = requests.get(
                f"{base_url}/api/v3/ticker/price",
                params={"symbol": symbol},
                timeout=10
            )
            response.raise_for_status()
            return float(response.json()["price"])
        except Exception as error:
            last_error = error

    raise RuntimeError(f"Could not get live price: {last_error}")


def check_entry_status(signal, entry, live_price, atr):
    """Classify whether price is near entry, has run away,
    or has moved against the setup. This is a warning, not a guarantee.
    """
    distance = ENTRY_NEAR_ATR * atr
    invalid_distance = ENTRY_INVALID_ATR * atr

    if signal == "BUY":
        if live_price >= entry + distance:
            return "MOVED AWAY - DO NOT CHASE"
        if live_price <= entry - invalid_distance:
            return "MOVED AGAINST SETUP - RECHECK"
    else:
        if live_price <= entry - distance:
            return "MOVED AWAY - DO NOT CHASE"
        if live_price >= entry + invalid_distance:
            return "MOVED AGAINST SETUP - RECHECK"

    return "ENTRY NEAR - VERIFY BEFORE ACTING"


def analyze_coin(symbol):
    try:
        h4 = add_indicators(get_candles(symbol, "4h"))
        h1 = add_indicators(get_candles(symbol, "1h"))
        m5 = add_indicators(get_candles(symbol, "5m"))

        main = trend_direction(h4)
        hourly = trend_direction(h1)

        if main not in ("BUY", "SELL"):
            return {
                "coin": symbol,
                "signal": "WAIT",
                "reason": "4H main trend is unclear"
            }

        last = m5.iloc[-1]
        prev = m5.iloc[-2]
        atr = float(last["atr"])

        if pd.isna(atr) or atr <= 0:
            return {
                "coin": symbol,
                "signal": "WAIT",
                "reason": "5M ATR is not ready"
            }

        close = float(last["close"])
        body = abs(float(last["close"] - last["open"]))
        earlier = m5.iloc[-(BREAKOUT_LOOKBACK + 1):-1]

        old_high = float(earlier["high"].max())
        old_low = float(earlier["low"].min())

        bullish = last["close"] > last["open"]
        bearish = last["close"] < last["open"]

        pullback_buy = (
            main == "BUY"
            and prev["low"] <= prev["ema20"] + 0.2 * atr
            and close > last["ema20"]
            and bullish
            and close > float(prev["high"])
        )

        pullback_sell = (
            main == "SELL"
            and prev["high"] >= prev["ema20"] - 0.2 * atr
            and close < last["ema20"]
            and bearish
            and close < float(prev["low"])
        )

        breakout_buy = (
            main == "BUY"
            and close > old_high
            and bullish
            and body >= MIN_BODY_ATR * atr
        )

        breakout_sell = (
            main == "SELL"
            and close < old_low
            and bearish
            and body >= MIN_BODY_ATR * atr
        )

        if main == "BUY" and hourly == "SELL":
            return {
                "coin": symbol,
                "signal": "WAIT",
                "reason": "1H strongly opposes 4H"
            }

        if main == "SELL" and hourly == "BUY":
            return {
                "coin": symbol,
                "signal": "WAIT",
                "reason": "1H strongly opposes 4H"
            }

        if pullback_buy or breakout_buy:
            signal = "BUY"
            setup_type = (
                "Pullback confirmation"
                if pullback_buy else "Breakout momentum"
            )
            stop = min(
                float(m5.iloc[-6:]["low"].min()),
                close - 0.5 * atr
            )
            risk = close - stop
            target = close + RISK_REWARD * risk

        elif pullback_sell or breakout_sell:
            signal = "SELL"
            setup_type = (
                "Pullback confirmation"
                if pullback_sell else "Breakout momentum"
            )
            stop = max(
                float(m5.iloc[-6:]["high"].max()),
                close + 0.5 * atr
            )
            risk = stop - close
            target = close - RISK_REWARD * risk

        else:
            return {
                "coin": symbol,
                "signal": "WAIT",
                "reason": f"No valid 5M entry; 4H={main}, 1H={hourly}"
            }

        if risk <= 0 or target <= 0:
            return {
                "coin": symbol,
                "signal": "WAIT",
                "reason": "Invalid stop-loss or target"
            }

        # Check the current quote against the planned entry.
        live_price = get_live_price(symbol)
        entry_status = check_entry_status(
            signal, close, live_price, atr
        )

        return {
            "coin": symbol,
            "signal": signal,
            "setup": f"{setup_type} | {entry_status}",
            "entry_status": entry_status,
            "setup_type": setup_type,
            "trend_4h": main,
            "trend_1h": hourly,
            "entry_reference": round(close, 8),
            "live_price": round(live_price, 8),
            "stop_loss_reference": round(stop, 8),
            "take_profit_reference": round(target, 8),
            "risk_reward": f"1:{RISK_REWARD}",
            "candle_time_utc": str(last["time"])
        }

    except Exception as error:
        return {
            "coin": symbol,
            "signal": "ERROR",
            "reason": str(error)
        }

In [ ]:

for coin in COINS:
    result = analyze_coin(coin)
    print(f"\n{coin}: {result['signal']}")
    print("Reason:", result.get("reason", result.get("setup", "")))
    if result.get("signal") in ("BUY", "SELL"):
        print("Entry:", result.get("entry_reference"))
        print("Live price:", result.get("live_price"))
        print("Entry status:", result.get("entry_status"))


BTCUSDT: WAIT
Reason: No valid 5M entry; 4H=SELL, 1H=MIXED

ETHUSDT: WAIT
Reason: No valid 5M entry; 4H=SELL, 1H=MIXED

SOLUSDT: WAIT
Reason: No valid 5M entry; 4H=SELL, 1H=MIXED


In [ ]:


import time
from datetime import datetime, timezone

SCAN_INTERVAL_SECONDS = 60

# Keep this set for the entire running session.
if "sent_alerts" not in globals():
    sent_alerts = set()

print("CRYPTO SIGNAL SCANNER STARTED")
print("Coins:", ", ".join(COINS))
print("Scan interval: 60 seconds")
print("Telegram alerts: ON")
print("Automatic trading: OFF")
print("To stop: tap the stop button on this Colab cell.")
print("-" * 40)

while True:
    now = datetime.now(timezone.utc)
    print(f"\nScan time (UTC): {now:%Y-%m-%d %H:%M:%S}")

    for coin in COINS:
        try:
            result = analyze_coin(coin)
            signal = result.get("signal", "ERROR")

            print(
                f"{coin}: {signal} — "
                f"{result.get('reason', result.get('setup', ''))}"
            )

            if signal not in ("BUY", "SELL"):
                continue

            candle_time = result.get("candle_time_utc", "")
            alert_key = (coin, signal, candle_time)

            if alert_key in sent_alerts:
                print(f"{coin}: duplicate alert skipped")
                continue

            send_signal_to_telegram(result)

        except Exception as error:
            print(f"{coin}: scan error — {error}")

    print(f"Next scan in {SCAN_INTERVAL_SECONDS} seconds.")
    time.sleep(SCAN_INTERVAL_SECONDS)

CRYPTO SIGNAL SCANNER STARTED
Coins: BTCUSDT, ETHUSDT, SOLUSDT
Scan interval: 60 seconds
Telegram alerts: ON
Automatic trading: OFF
To stop: tap the stop button on this Colab cell.
----------------------------------------

Scan time (UTC): 2026-10-10 08:08:24
BTCUSDT: WAIT — No valid 5M entry; 4H=SELL, 1H=MIXED
ETHUSDT: WAIT — No valid 5M entry; 4H=SELL, 1H=MIXED
SOLUSDT: WAIT — No valid 5M entry; 4H=SELL, 1H=MIXED
Next scan in 60 seconds.

Scan time (UTC): 2026-10-10 08:09:29
BTCUSDT: WAIT — No valid 5M entry; 4H=SELL, 1H=MIXED
ETHUSDT: WAIT — No valid 5M entry; 4H=SELL, 1H=MIXED
SOLUSDT: WAIT — No valid 5M entry; 4H=SELL, 1H=MIXED
Next scan in 60 seconds.

Scan time (UTC): 2026-10-10 08:10:34
BTCUSDT: WAIT — No valid 5M entry; 4H=SELL, 1H=MIXED
ETHUSDT: WAIT — No valid 5M entry; 4H=SELL, 1H=MIXED
SOLUSDT: WAIT — No valid 5M entry; 4H=SELL, 1H=MIXED
Next scan in 60 seconds.

Scan time (UTC): 2026-10-10 08:11:38
BTCUSDT: WAIT — No valid 5M entry; 4H=SELL, 1H=MIXED
ETHUSDT: WAIT — No va

KeyboardInterrupt: 

In [ ]:


import time
from datetime import datetime, timezone

print("🚀 Crypto scanner started")
print("Coins:", ", ".join(COINS))
print("Scan interval: 60 seconds")
print("Automatic trading: OFF")
print("Tap Stop to pause scanning.")

while True:
    print("\n" + "=" * 35)
    print("Scan time (UTC):", datetime.now(timezone.utc).strftime("%Y-%m-%d %H:%M:%S"))

    for coin in COINS:
        try:
            result = analyze_coin(coin)
            signal = result.get("signal", "WAIT")
            reason = result.get("reason", result.get("setup", ""))

            print(f"{coin}: {signal} — {reason}")

            if signal in ("BUY", "SELL"):
                send_signal_to_telegram(result)

        except Exception as error:
            print(f"{coin}: Scan error — {error}")

    print("Scan complete. Next scan in 60 seconds.")
    time.sleep(60)

🚀 Crypto scanner started
Coins: BTCUSDT, ETHUSDT, SOLUSDT
Scan interval: 60 seconds
Automatic trading: OFF
Tap Stop to pause scanning.

Scan time (UTC): 2026-10-10 09:30:08
BTCUSDT: WAIT — No valid 5M entry; 4H=SELL, 1H=MIXED
ETHUSDT: WAIT — No valid 5M entry; 4H=SELL, 1H=MIXED
SOLUSDT: WAIT — No valid 5M entry; 4H=SELL, 1H=MIXED
Scan complete. Next scan in 60 seconds.

Scan time (UTC): 2026-10-10 09:31:12
BTCUSDT: WAIT — No valid 5M entry; 4H=SELL, 1H=MIXED
ETHUSDT: WAIT — No valid 5M entry; 4H=SELL, 1H=MIXED
SOLUSDT: WAIT — No valid 5M entry; 4H=SELL, 1H=MIXED
Scan complete. Next scan in 60 seconds.

Scan time (UTC): 2026-10-10 09:32:17
BTCUSDT: WAIT — No valid 5M entry; 4H=SELL, 1H=MIXED
ETHUSDT: WAIT — No valid 5M entry; 4H=SELL, 1H=MIXED
SOLUSDT: WAIT — No valid 5M entry; 4H=SELL, 1H=MIXED
Scan complete. Next scan in 60 seconds.

Scan time (UTC): 2026-10-10 09:33:22
BTCUSDT: WAIT — No valid 5M entry; 4H=SELL, 1H=MIXED
ETHUSDT: WAIT — No valid 5M entry; 4H=SELL, 1H=MIXED
SOLUSDT: W